In [1]:
import pandas as pd

df = pd.read_csv("dataset_famybot_train_v1.csv")

print("Filas:", len(df))
print("Columnas:", list(df.columns))

df.head()

Filas: 1569
Columnas: ['texto', 'intencion', 'fuente', 'version']


,texto,intencion,fuente,version
0,hola que tal,saludo,sintetico,v1
1,buenas?,saludo,sintetico,v1
2,quisiera saber saludos,saludo,sintetico,v1
3,que tal gracias,saludo,sintetico,v1
4,holaaa,saludo,sintetico,v1


In [2]:
df["intencion"].value_counts()

,count
intencion,
consultar_ubicacion,111
trabajo,111
saludo,109
agendar_cita,108
consultar_horario,107
consulta_especialidades,107
cotizar_servicio,107
consulta_servicios,106
hablar_asesor,103


In [3]:
from sklearn.model_selection import train_test_split

X = df["texto"]
y = df["intencion"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Entrenamiento:", len(X_train))
print("Prueba:", len(X_test))

Entrenamiento: 1255
Prueba: 314


In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1,2)
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print("Entrenamiento:", X_train_tfidf.shape)
print("Prueba:", X_test_tfidf.shape)

Entrenamiento: (1255, 1133)
Prueba: (314, 1133)


In [5]:
from sklearn.svm import LinearSVC

model = LinearSVC()

model.fit(X_train_tfidf, y_train)

print("Modelo entrenado correctamente")

Modelo entrenado correctamente


In [6]:
from sklearn.metrics import accuracy_score

y_pred = model.predict(X_test_tfidf)

accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", round(accuracy * 100, 2), "%")

Accuracy: 97.77 %


In [7]:
pruebas = [
    "quiero una cita con cardiologia",
    "cuanto cuesta una resonancia",
    "tienen ginecologia",
    "donde estan ubicados",
    "quiero enviar mi hoja de vida",
    "gracias por la informacion",
    "hola",
    "realizan tomografias",
    "quiero hablar con una persona",
    "atienden los sabados"
]

pruebas_tfidf = vectorizer.transform(pruebas)

predicciones = model.predict(pruebas_tfidf)

for texto, pred in zip(pruebas, predicciones):
    print(f"{texto} -> {pred}")

quiero una cita con cardiologia -> agendar_cita
cuanto cuesta una resonancia -> cotizar_servicio
tienen ginecologia -> consulta_especialidades
donde estan ubicados -> consultar_ubicacion
quiero enviar mi hoja de vida -> trabajo
gracias por la informacion -> agradecimiento
hola -> saludo
realizan tomografias -> consulta_servicios
quiero hablar con una persona -> hablar_asesor
atienden los sabados -> consultar_horario


In [9]:
pruebas_reales = [
    "eco abdominal precio",
    "hay pediatra",
    "necesito un turno",
    "fonde quedan",
    "quiero mandar mi cv",
    "hacen resonancia contrastada",
    "horario de atencion",
    "quiero resultados de laboratorio"
]

pruebas_reales_tfidf = vectorizer.transform(pruebas_reales)

predicciones_reales = model.predict(pruebas_reales_tfidf)

for texto, pred in zip(pruebas_reales, predicciones_reales):
    print(f"{texto} -> {pred}")

eco abdominal precio -> cotizar_servicio
hay pediatra -> consulta_especialidades
necesito un turno -> agendar_cita
fonde quedan -> consultar_ubicacion
quiero mandar mi cv -> trabajo
hacen resonancia contrastada -> consulta_servicios
horario de atencion -> consultar_horario
quiero resultados de laboratorio -> consultar_resultados


In [10]:
casos_basura = [
    "mi gato no quiere comer",
    "como esta el clima hoy",
    "quiero comprar una moto",
    "quien gano el partido",
    "me recomiendas una pelicula",
    "receta para lasagna",
    "quiero aprender ingles",
    "cuanto cuesta un iphone"
]

basura_tfidf = vectorizer.transform(casos_basura)

pred_basura = model.predict(basura_tfidf)

for texto, pred in zip(casos_basura, pred_basura):
    print(f"{texto} -> {pred}")

mi gato no quiere comer -> desconocido
como esta el clima hoy -> desconocido
quiero comprar una moto -> desconocido
quien gano el partido -> desconocido
me recomiendas una pelicula -> agendar_cita
receta para lasagna -> desconocido
quiero aprender ingles -> desconocido
cuanto cuesta un iphone -> cotizar_servicio


In [11]:
from sklearn.linear_model import LogisticRegression

model_lr = LogisticRegression(
    max_iter=1000
)

model_lr.fit(X_train_tfidf, y_train)

print("Modelo LogisticRegression entrenado correctamente")

Modelo LogisticRegression entrenado correctamente


In [12]:
texto_prueba = [
    "cuanto cuesta un iphone",
    "me recomiendas una pelicula",
    "quiero una cita con cardiologia",
    "hay pediatra",
    "fonde quedan"
]

X_demo = vectorizer.transform(texto_prueba)

predicciones = model_lr.predict(X_demo)
probabilidades = model_lr.predict_proba(X_demo)

for texto, pred, probs in zip(texto_prueba, predicciones, probabilidades):
    confianza = probs.max()

    print(f"Texto: {texto}")
    print(f"Predicción: {pred}")
    print(f"Confianza: {confianza:.4f}")
    print("-" * 50)

Texto: cuanto cuesta un iphone
Predicción: cotizar_servicio
Confianza: 0.5650
--------------------------------------------------
Texto: me recomiendas una pelicula
Predicción: agendar_cita
Confianza: 0.3159
--------------------------------------------------
Texto: quiero una cita con cardiologia
Predicción: agendar_cita
Confianza: 0.6232
--------------------------------------------------
Texto: hay pediatra
Predicción: consulta_especialidades
Confianza: 0.5539
--------------------------------------------------
Texto: fonde quedan
Predicción: consultar_ubicacion
Confianza: 0.2320
--------------------------------------------------


In [13]:
import numpy as np

texto_prueba = [
    "hay pediatra",
    "fonde quedan",
    "cuanto cuesta un iphone",
    "me recomiendas una pelicula"
]

X_demo = vectorizer.transform(texto_prueba)

probabilidades = model_lr.predict_proba(X_demo)

for texto, probs in zip(texto_prueba, probabilidades):

    top3_idx = np.argsort(probs)[-3:][::-1]

    print(f"\nTexto: {texto}")

    for idx in top3_idx:
        print(
            f"{model_lr.classes_[idx]} -> {probs[idx]:.4f}"
        )


Texto: hay pediatra
consulta_especialidades -> 0.5539
consultar_promociones -> 0.1667
trabajo -> 0.0271

Texto: fonde quedan
consultar_ubicacion -> 0.2320
trabajo -> 0.0744
consulta_especialidades -> 0.0619

Texto: cuanto cuesta un iphone
cotizar_servicio -> 0.5650
desconocido -> 0.0559
trabajo -> 0.0404

Texto: me recomiendas una pelicula
agendar_cita -> 0.3159
cotizar_servicio -> 0.1022
trabajo -> 0.0881


In [14]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

                         precision    recall  f1-score   support

           agendar_cita       0.96      1.00      0.98        22
         agradecimiento       1.00      1.00      1.00        20
               alianzas       0.95      1.00      0.98        20
consulta_especialidades       1.00      0.95      0.98        22
     consulta_servicios       1.00      0.95      0.98        21
      consultar_horario       0.95      1.00      0.98        21
  consultar_promociones       1.00      1.00      1.00        20
   consultar_resultados       1.00      1.00      1.00        20
    consultar_ubicacion       1.00      0.95      0.98        22
       cotizar_servicio       1.00      1.00      1.00        21
            desconocido       1.00      1.00      1.00        20
          hablar_asesor       1.00      1.00      1.00        21
            proveedores       0.95      1.00      0.98        20
                 saludo       0.95      0.95      0.95        22
                trabajo 

In [15]:
import joblib

joblib.dump(vectorizer, "vectorizer_famybot_v1.pkl")
joblib.dump(model, "classifier_famybot_v1.pkl")

print("Archivos guardados correctamente")

Archivos guardados correctamente
